# 3 - Haversine Speed Derivation and Savitzky-Golay Smoothing

Fills in the missing `speed` field of the Endomondo running workouts: for each workout, the distance between consecutive GPS points is computed with the haversine formula, divided by the time difference to get speed in km/h, cleaned of data quality problems (repeated timestamps, extreme values) and smoothed with a Savitzky-Golay filter. The method is developed and validated on a single workout first, then applied to all rows.

**Inputs:** `data/endomondoHR_proper.csv` (output of the earlier preprocessing notebooks; array columns stored as text). It is also overwritten with the parsed version as an intermediate save. Helper functions are not imported from `functions.py` in this notebook.

**Outputs:** `data/endomondoHR_speed.csv` (all workouts with a filled `speed` array). This is the input of the later steps (e.g. the aerobic/anaerobic labelling, where the labels `aerobik`/`anaerobik` are intentionally Turkish-named because they are part of the API contract).

Part of RunSight - see the repo README.

# Deriving Missing Speed from GPS

## Problem

As we saw in the previous notebook (`1_data_preprocess.ipynb`), only about
11,500 of the 70,591 running workouts have a directly recorded `speed`
field; in the remaining ~60,000 workouts it is empty. However, every
workout does have GPS positions (latitude/longitude) and a timestamp for
each measurement. Together these are enough to compute speed: how far a
person moved between two GPS points and how long it took -- distance / time = speed.

## What this notebook does (summary)

1. **Distance computation (haversine formula)** -- a mathematical formula
   that gives the real distance, along the surface of a sphere, between
   two GPS coordinates in meters.
2. **Raw speed derivation** -- dividing the distance between consecutive
   points by the time between them to get speed in km/h.
3. **Fixing data quality problems** -- GPS and timestamp data are not
   perfect (repeated timestamps, signal jumps); without correcting them,
   the raw speed is too noisy to use.
4. **Smoothing (Savitzky-Golay filter)** -- suppressing the remaining
   small noise while preserving the real speed changes
   (acceleration/deceleration), to obtain a smooth speed curve.

The first half of this notebook develops and validates the method step by
step on a SINGLE workout; in the second half the same method is applied to
all 70,591 rows and the result is saved.

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
PATH = "data/endomondoHR_proper.csv"

df = pd.read_csv(PATH) 

## Getting to know the data: what the columns look like in the CSV

To use the haversine function, we first need to see what kind of data the
`latitude`/`longitude` columns actually contain. In the `df.info()` output
we saw that these columns are `str` (text) -- when CSV saves Python lists
it converts them to plain text (like `"[52.22, 52.23, ...]"`), and when
reading, pandas does not automatically turn this back into a numeric array.
The next few cells confirm this: the `speed` cell is an empty list string
(`'[]'`), and the `latitude` cell is a long string (`len()` returns 5576,
which is the number of characters, not the actual number of points).

In [ ]:
df.info()

In [ ]:
df['speed'][0]

In [ ]:
type(df['speed'][0])

In [ ]:
len(df['latitude'][0])

## Converting text to a numeric array

We need to convert a string like `"[52.22, 52.23, ...]"` into a real numeric
array (numpy array) -- otherwise we cannot do math on it. The method is
simple: delete the square brackets (`[`, `]`), replace commas with spaces,
and read the remaining space-separated numbers in one go with
`np.fromstring`. The `arr[0:10]` output confirms that the conversion really
works and yields actual latitude values (like 52.22...).

In [ ]:
cleaned = df['latitude'][0].replace("[", "").replace("]", "").replace(",", " ")
arr = np.fromstring(cleaned, sep=" ", dtype=np.float64)

In [ ]:
arr[0:10]

In [ ]:
import ast 

## Applying this conversion to all relevant columns

We wrap the conversion above in a one-line function (`parse_arr`) so that we
can apply the same operation to ALL of the `longitude`, `altitude`,
`latitude`, `heart_rate`, `timestamp` and `speed` columns without rewriting
it each time (the `num_cols` list + `.apply(parse_arr)`). After it finishes,
`df.info()` confirms that these columns are now of type `object` (i.e. real
Python objects -- numpy arrays here); non-numeric columns such as `sport`
and `gender` should stay as text, so we leave them untouched.

In [ ]:
def parse_arr(s):
    if isinstance(s, np.ndarray):
        return s.astype(np.float64)
    cleaned = s.replace("[", "").replace("]", "").replace(",", " ")
    return np.fromstring(cleaned, sep=" ", dtype=np.float64)


In [ ]:
df.columns

In [ ]:
num_cols = ['longitude', 'altitude', 'latitude','heart_rate', 'timestamp', 'speed']

In [ ]:
for col in num_cols:
    df[col] = df[col].apply(parse_arr)

In [ ]:
df.info()

In [ ]:
df[num_cols].head()

## Intermediate save

The parse step (string -> array) takes a while for the whole dataset; to
avoid repeating it in every session, we save the result over the same CSV
file. Note: the CSV format stores arrays as text again (there is no way to
write a numpy array directly to CSV) -- so when we read the file back we
will still need to run the `parse_arr` step, but at least we do not have to
repeat all the earlier steps from the raw JSON up to this point (download,
parsing, filtering).

In [ ]:
import os
os.makedirs("data", exist_ok=True)
df.to_csv(PATH)

## The haversine formula -- how to compute the distance between two GPS points

### Why we cannot use a simple "straight-line distance" (Euclidean distance)

Latitude and longitude describe a point on the Earth's surface in degrees.
To find the distance between two points you might want to use a flat
(Euclidean) formula such as "the square root of the squared latitude
difference plus the squared longitude difference" -- but this gives the
WRONG result, because:

1. **The Earth is not flat, it is a sphere** -- latitude/longitude degrees
   are not a flat coordinate system but angular positions on the surface of
   a sphere. On a sphere's surface, the "shortest path" is not a straight
   line but an arc (a curve along a great circle).
2. **1 degree of longitude does not correspond to the same distance
   everywhere** -- at the equator 1° of longitude corresponds to ~111 km,
   but as you approach the poles the meridians converge and the same 1° of
   longitude corresponds to much less physical distance (it drops to zero
   at the pole). 1° of latitude, on the other hand, is about the same
   distance everywhere (~111 km). So the latitude and longitude axes are not
   "equally weighted" -- the real distance of a longitude difference depends
   on the latitude you are at.

The haversine formula is a mathematically correct distance formula for the
surface of a sphere that takes exactly this into account.

### The formula, step by step

```python
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000.0  # Earth radius (meters)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
```

- **`R = 6371000.0`** -- the mean radius of the Earth, in meters. The basic
  scale needed to get an arc length on the sphere.
- **`np.radians(...)`** -- trigonometric functions (`sin`, `cos`) expect
  angles in radians, but GPS coordinates are given in degrees (e.g.
  latitude 52.22°). This line converts degrees to radians (degrees × π/180).
- **`phi1, phi2`** -- the latitudes of the two points (in radians); in the
  geographic literature latitude is conventionally denoted by the Greek
  letter φ (phi).
- **`dphi`, `dlambda`** -- the latitude and longitude differences between
  the two points (in radians); the letter λ (lambda) is used for the
  longitude difference.
- **`a = sin²(Δφ/2) + cos(φ1)·cos(φ2)·sin²(Δλ/2)`** -- this is the
  "haversine" (half-versine) function itself. The `cos(φ1)·cos(φ2)` factor
  is the part that formalizes the fact mentioned above that "the real
  distance of longitude depends on latitude" -- longitude differences near
  the equator (large `cos`) get more weight, and those near the poles (small
  `cos`) get less weight.
- **`2 * R * arctan2(√a, √(1-a))`** -- converts the value `a` into the real
  arc length on the sphere (meters). Using `arctan2` (instead of plain
  `arcsin`) gives a more numerically stable result, especially for very
  close or very distant points.

**In short:** this formula takes two latitude/longitude pairs and returns,
in meters, not "the distance a bird would cover flying in a straight line"
but "the shortest-path distance along the surface of the Earth's sphere" --
which is exactly what GPS-based speed calculations need.

In [ ]:
def haversine_np(lat1, lon1, lat2, lon2):
    """Computes the distance (meters) between consecutive GPS coordinates."""
    R = 6371000.0  # Earth radius (meters)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

## Trying haversine on a single pair of points

Instead of applying a complex function directly to all 70,591 rows, we first
try it on the FIRST TWO GPS points of a SINGLE workout. This lets us quickly
verify that the function works, and also easily answer "is the result
plausible?": the value `9.94` in the cell below is in METERS -- i.e. there
is a distance of ~10 meters between these two GPS points, which is exactly
the order of magnitude expected between two consecutive GPS samples taken a
few seconds apart.

In [ ]:
lat1 = df['latitude'][0][0]
lat2 = df['latitude'][0][1]
lon1 = df['longitude'][0][0]
lon2 = df['longitude'][0][1]


In [ ]:
speed = haversine_np(lat1,lon1,lat2,lon2)

In [ ]:
speed

## Is there a single-point jump in the GPS signal?

In the real world, a GPS measurement can occasionally give a wrong position
(due to building/tree shadows, signal reflections, etc.) -- this makes a
single point "jump" away from the true position. Such a point produces a
huge, unrealistic spike in the speed computation (it looks as if the person
suddenly covered hundreds of meters per second).

The logic for detecting it is simple but effective: if point `i` is truly
corrupted, the speed from `i-1 → i` comes out abnormally high, and the speed
from `i → i+1` also comes out abnormally high -- BUT if we SKIP point `i`
entirely and compute the speed from `i-1 → i+1`, it returns to a normal
value (because the real position did not change; only point `i` was
recorded incorrectly). In other words, "fast before, fast after, but normal
when skipped" -- by comparing exactly these three, we decide whether point
`i` is corrupted. `MAX_MPS` (the m/s equivalent of 38 km/h -- an elite
sprint ceiling) is used here as the physical limit for "abnormally fast".

In [ ]:
row = df.iloc[0]
lat, lon, ts = row["latitude"], row["longitude"], row["timestamp"]

MAX_SPEED_KMH = 38.0                 # elite sprint ceiling (~10.5 m/s)
MAX_MPS = MAX_SPEED_KMH / 3.6

d_direct = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])   # meters, i -> i+1
dt_direct = np.diff(ts)
dt_direct = np.where(dt_direct <= 0, 1e6, dt_direct)             # dt=0 is a separate problem, don't count it as "fast"

d_skip = haversine_np(lat[:-2], lon[:-2], lat[2:], lon[2:])     # meters, i -> i+2 (skipping i)
dt_skip = ts[2:] - ts[:-2]
dt_skip = np.where(dt_skip <= 0, 1e6, dt_skip)

fast_in = (d_direct[:-1] / dt_direct[:-1]) > MAX_MPS    # is i-1 -> i fast?
fast_out = (d_direct[1:] / dt_direct[1:]) > MAX_MPS     # is i -> i+1 fast?
skip_ok = (d_skip / dt_skip) <= MAX_MPS                  # is it normal when skipping i (i-1 -> i+1)?

spike = np.zeros(len(lat), dtype=bool)
spike[1:-1] = fast_in & fast_out & skip_ok   # fast both before and after, but normal when skipped -> a single corrupted point

print("number of suspicious GPS points:", spike.sum(), "/", len(lat))
print("indices:", np.where(spike)[0][:15])


## Putting everything into a single function

We combine the steps we have tried piece by piece so far (distance
computation → raw speed → data quality correction → smoothing) into a single
function (`compute_speed_kmh`). Each step inside the function is explained
below.

### 1) Consecutive distance and time difference

```python
dist = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])
dt = np.diff(ts)
```

`lat[:-1]` is the first N-1 elements of the array and `lat[1:]` is elements
2 through N -- passing both to haversine together makes `dist[i]` = the
distance (meters) between point `i` and point `i+1`. `np.diff(ts)` gives the
consecutive time differences (seconds) in the same way. Both have length N-1
(N points have N-1 "intervals" between them).

### 2) Raw speed = distance / time -- but there is a division-by-zero trap

```python
valid = dt > 0
speed = np.where(valid, (dist / np.where(valid, dt, 1)) * 3.6, np.nan)
```

Speed = distance/time, then `× 3.6` converts from m/s to km/h. But GPS data
frequently contains **multiple measurements for the same second** (dt=0,
repeated timestamp) -- about 22% in this dataset. Dividing by zero is
mathematically undefined (it gives `inf`/an error). This line applies a
two-layer protection: `np.where(valid, dt, 1)` temporarily puts `1` where
`dt=0` DURING the division (only so the computation does not crash -- we
discard this value in the next step anyway); the outer `np.where` writes
`NaN` ("undefined") into the actual result wherever `dt<=0`.

### 3) Estimate undefined points from their neighbors (interpolation)

```python
idx = np.arange(len(speed))
nan_mask = np.isnan(speed)
if nan_mask.any() and (~nan_mask).any():
    speed[nan_mask] = np.interp(idx[nan_mask], idx[~nan_mask], speed[~nan_mask])
```

At points where `dt=0`, speed is genuinely undefined -- putting an arbitrary
or artificially large number there (in our first attempts we used `0.001`
instead of `dt=0`) caused fake, huge speed spikes (we saw absurd values such
as 155,000 km/h). The right solution: leave these points empty (`NaN`) and
do **linear interpolation** between the nearest VALID (non-NaN) speed values
to the left and right -- i.e. estimate "a sensible intermediate value
between these two real measurements". `np.interp` does exactly this.

### 4) Align the length with the other columns

```python
speed = np.insert(speed, 0, speed[0])
```

The `speed` array always has length N-1 (because it is computed between
consecutive PAIRS). But we need one speed value per GPS point (length N) --
so that it is aligned row by row with the other columns of the DataFrame
(lat, hr, etc.). The simplest solution: pad the array to N by inserting the
first point's speed once more at the beginning.

### 5) Clip non-physical extreme values

```python
speed = np.clip(speed, 0.0, max_speed_kmh)
```

`max_speed_kmh=38.0` km/h -- a ceiling that even an elite sprinter could not
exceed (~10.5 m/s). A single bad GPS reading can still produce an
instantaneous extreme speed; this line caps such extremes at a reasonable
upper bound, while also keeping negative speed (physically meaningless) at
the lower bound (0).

### 6) Smoothing with the Savitzky-Golay filter

```python
if len(speed) >= window_length and window_length > polyorder:
    speed = savgol_filter(speed, window_length, polyorder)
    speed = np.clip(speed, 0.0, max_speed_kmh)
```

Even after the steps so far, the speed data is still "rough" -- because the
GPS's own measurement precision is limited, even for someone running at a
constant pace the speed value jumps slightly from sample to sample. The
Savitzky-Golay filter is a special smoothing method used to suppress this
roughness.

**How it works:** around each target point it takes `window_length` (7
here) neighboring samples (3 before, the point itself, 3 after), fits a
polynomial curve of degree `polyorder` (2 here, i.e. a parabola) to these 7
points (by least squares), and then reads the value of that curve at the
EXACT CENTER OF THE WINDOW. The window slides to the next point and the same
process repeats.

**Why this and not a plain moving average:** a plain average sums all points
in the window with equal weight and divides -- this also flattens away the
real acceleration/deceleration trend (for example it blunts the sharp rise
at the start of a sprint). Savitzky-Golay fits a CURVE to the neighboring
points, so it both suppresses noise and largely preserves the slope/
acceleration (i.e. the real acceleration trend) -- which is why it is more
often preferred over a moving average for time-series smoothing.

**Rules for the two parameters:**
- `window_length` **must be an odd number** -- so that the "exact center of
  the window" is a well-defined point (3 before + the point itself + 3 after
  = symmetric 7; with an even number there would be two middle points, which
  is ambiguous).
- `window_length` **must be greater than `polyorder`** -- fitting a
  polynomial curve uniquely requires at least `polyorder+1` points (at least
  3 points for a 2nd-degree parabola).

The `if` condition in the function silently skips the filter when these two
rules are violated (a very short workout, or an invalid parameter
combination), preventing the computation from crashing. Applying `clip`
again after savgol is a safety margin -- since we fixed the dt=0 problem at
its root we no longer expect serious overshoot, but there is no guarantee.

In [ ]:
def compute_speed_kmh(lat, lon, ts, window_length=7, polyorder=2, max_speed_kmh=38.0):
    """
    Derives speed (km/h) from GPS + time, interpolates dt=0 points,
    clips extreme values and smooths with Savitzky-Golay.
    """

    # delta phi (latitude difference) lat[:-1], lat[1:] and delta lambda (longitude difference) lon[:-1], lon[1:]
    """lat[:-1] and lat[1:]: one runs from the start up to N-1, the other from the 2nd element to N.
       Their difference = consecutive point pairs. dist[i] = distance (meters) between point i and i+1.
       dt[i] is the time difference (seconds) for the same pair. Both have length N-1."""
    
    dist = haversine_np(lat[:-1], lon[:-1], lat[1:], lon[1:])
    dt = np.diff(ts)

    """ In our experiments we found dt=0 (repeated timestamp) in ~22% of the rows.
        Dividing by dt=0 gives inf/an error. np.where(valid, dt, 1) temporarily puts 1 where dt=0 during the division (only so it does not crash -- we will not use that result anyway);
        the outer np.where writes NaN ("speed is undefined here") into the actual result wherever dt<=0. *3.6: m/s -> km/h. """
    valid = dt > 0
    speed = np.where(valid, (dist / np.where(valid, dt, 1)) * 3.6, np.nan)


    """ Interpolate the NaNs from their neighbors"""
    idx = np.arange(len(speed))
    nan_mask = np.isnan(speed)
    if nan_mask.any() and (~nan_mask).any():
        speed[nan_mask] = np.interp(idx[nan_mask], idx[~nan_mask], speed[~nan_mask])

    """ speed always has length N-1 (because it is computed between consecutive PAIRS).
        We pad it to N by inserting the first point's speed once more at the beginning --
        so that it is aligned with the DataFrame's other columns (lat, hr...). """
    
    speed = np.insert(speed, 0, speed[0])         

    """ max_speed_kmh=38.0: elite sprint ceiling.
        A single bad GPS reading can produce an instantaneous huge speed; the lower bound also prevents negative speed. """
    
    speed = np.clip(speed, 0.0, max_speed_kmh)      

    """ polyorder is generally 2-3 by convention; it sets the degree of the fitted polynomial.
        In a sliding window of window_length=7 points it fits polyorder=2 (a parabola) and reads the value at the center of the window from that curve
        -- unlike a plain moving average, it preserves acceleration/deceleration.
        if: skip the filter if the window is larger than the data length or the degree is not smaller than the window (mathematically impossible).
        Clip again after savgol: since we fixed the dt=0 problem at its root we no longer expect serious overshoot, but it stays as a safety margin.
        It opens a sliding window for each point: window_length=7 means
        taking the 7 neighboring samples around each target point (3 before, the point itself, 3 after) and fitting a polynomial of degree polyorder (by least squares) to them,
        then reading that polynomial's value at the exact center of the window. Then the window slides one step to the right and the same process repeats.

        There are two hard constraints:

        It must be odd -- because the "center of the window" must be a well-defined point (3 before + the point itself + 3 after = symmetric). With an even number there would be two middle points, which is ambiguous.
        It must be greater than polyorder -- to fit a polynomial uniquely you need at least polyorder+1 points (at least 3 points for a 2nd-degree parabola).
  """

    if len(speed) >= window_length and window_length > polyorder:
        speed = savgol_filter(speed, window_length, polyorder)
        speed = np.clip(speed, 0.0, max_speed_kmh)

    return speed




# verify in one line -- should be consistent with the single-workout result above (min/max, length)
test = compute_speed_kmh(lat, lon, ts)
print("length:", len(test), "min/max:", test.min(), test.max())
print("first 5:", np.round(test[:5], 2))

## Critical Note
There is something here that is easy to overlook but important: window_length=7 is fixed in terms of samples (index), but the samples are not evenly spaced in real time. The dataset resamples every workout to exactly 500 points, regardless of its duration (this is also mentioned in the note of run_clustering_aerobic_anaerobic.py itself: "in a 2-hour run the points are ~14 s apart, in a 30-minute one ~3.6 s apart").

As a result:

In a 2-hour run: 7 samples × ~14 s ≈ smoothing over a time window of about 98 seconds.
In a 30-minute run: 7 samples × ~3.6 s ≈ a window of about 25 seconds.
So the same window_length=7 applies a very different width of smoothing in real-world terms depending on the workout duration -- in long runs you are blending a time span ~4 times wider. In a short, intense interval workout, the 98-second-equivalent window (as in long runs) would probably flatten away the entire interval structure if applied to that short workout; but since the duration is different, it behaves differently. This is a point to be careful about later (especially in finer analyses such as interval/tempo separation), showing that the assumption "a fixed window_length treats every workout fairly" is actually wrong -- it does not cause a problem for the current simple goal of "fill in the missing speed", but it must be used knowingly.

## One last visual/numerical check

We do one more sanity check on the speed values the function produced: are
the values in a reasonable range (no extreme spikes), and are there sudden,
meaningless jumps between consecutive values? We print the first 50 values
and inspect them by eye -- this is the last confidence check before the
upcoming "apply to the whole dataset" step.

In [ ]:
print("first 50:", np.round(test[:50], 2))

## From a single row to the whole dataset

After developing and validating `compute_speed_kmh` on a single workout, we
can now safely apply it to all 70,591 rows. We pass each row's `latitude`,
`longitude` and `timestamp` arrays to the function and write the returned
speed array into the `speed` column -- this means the ~60,000 rows that were
initially empty (`[]`) are now filled as well. The `len(df['speed'].iloc[0])`
and `.apply(len).gt(0).all()` checks confirm that no row is left empty and
that all arrays have the expected length (500).

In [ ]:
df["speed"] = [
    compute_speed_kmh(la, lo, t)
    for la, lo, t in zip(df["latitude"], df["longitude"], df["timestamp"])
]

print("sample length:", len(df['speed'].iloc[0]))
print("are all rows filled:", df['speed'].apply(len).gt(0).all())

In [ ]:
df['speed']

## Saving the result

All workouts now have speed data; we save it to a separate file
(`endomondoHR_speed.csv`) -- we do NOT overwrite the original
`endomondoHR_proper.csv`, so the state before the speed data is preserved as
well. This new file will be the main data source in the next step of the
project (`5_polarize.ipynb` -- aerobic/anaerobic classification).

In [ ]:
import os
os.makedirs("data", exist_ok=True)
df.to_csv('data/endomondoHR_speed.csv')